In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO
import torch
%matplotlib inline

### Get the best weight for each model

In [2]:
cwd = os.getcwd()
yolo_dirs = ['yolo8', 'yolo11', 'yolo9', 'yolo12']
gen_dir = [os.path.join(cwd, gen) for gen in yolo_dirs]
models_name = [os.listdir(model_dir) for model_dir in gen_dir if os.path.isdir(model_dir)]

model_dirs = []
for gen, models in zip(gen_dir, models_name):
    for model in models:
        path = os.path.join(gen, model)
        if os.path.isdir(path):
            model_dirs.append(path)
        
models_weight_path = [os.path.join(model_dir, "weights", "best.pt") for model_dir in model_dirs]
models_weight_path


['/home/kien/Desktop/projects/Amir_research/yolo/yolo8/yolov8m/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo8/yolov8n/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo8/yolov8l/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo8/yolov8x/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo8/yolov8s/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo11/yolo11l/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo11/yolo11n/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo11/yolo11s/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo11/yolo11m/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo11/yolo11x/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo9/yolov9e2/weights/best.pt',
 '/home/kien/Desktop/projects/Amir_research/yolo/yolo9/yolov9s/weights/best.pt',
 '/home/kien/Desktop/p

### Load all yolo models into DF

In [3]:
from collections import defaultdict
dic = defaultdict(list)

for model_path in models_weight_path:
    
    str_split = model_path.split("/")
    dic["gen"].append(str_split[7])
    dic["name"].append(str_split[8])
    dic["model"].append(YOLO(model_path))

    
df = pd.DataFrame(dic)
df


,gen,name,model
0,yolo8,yolov8m,YOLO(\n (model): DetectionModel(\n (model)...
1,yolo8,yolov8n,YOLO(\n (model): DetectionModel(\n (model)...
2,yolo8,yolov8l,YOLO(\n (model): DetectionModel(\n (model)...
3,yolo8,yolov8x,YOLO(\n (model): DetectionModel(\n (model)...
4,yolo8,yolov8s,YOLO(\n (model): DetectionModel(\n (model)...
5,yolo11,yolo11l,YOLO(\n (model): DetectionModel(\n (model)...
6,yolo11,yolo11n,YOLO(\n (model): DetectionModel(\n (model)...
7,yolo11,yolo11s,YOLO(\n (model): DetectionModel(\n (model)...
8,yolo11,yolo11m,YOLO(\n (model): DetectionModel(\n (model)...
9,yolo11,yolo11x,YOLO(\n (model): DetectionModel(\n (model)...


### EVAL

In [4]:
import os, gc, torch

results = []
data_dir = os.path.join("..","pretrain_models","data","mobile detection.v7i.yolov11","data.yaml")

VAL_KW = dict(
    data=data_dir,
    split="test",
    imgsz=640,
    iou=0.50,
    conf=0.001,
    batch=1,
    device=0,
    half=True,      
    workers=0,
    verbose=False,
)


with torch.no_grad():
    for i, yolo_obj in df["model"].items():
        try:
            # Make sure it's on GPU only while we need it
            yolo_obj.model.to("cuda").eval()

            res = yolo_obj.val(**VAL_KW)

            results.append(res)
        finally:
            # Move OFF GPU, break all refs, and clear caches
            try:
                yolo_obj.model.to("cpu")
            except Exception:
                pass
            df.at[i, "model"] = None    # <- break the DF reference
            del yolo_obj
            gc.collect()
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

Ultralytics 8.3.195 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3080, 9864MiB)
Model summary (fused): 92 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1641.7±587.1 MB/s, size: 39.9 KB)
val: Scanning /home/kien/Desktop/projects/Amir_research/pretrain_models/data/mobile detection.v7i.yolov11/test/labels.cache... 270 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 270/270 619.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 270/270 75.3it/s 3.6s0.1s
                   all        270        271      0.959      0.926      0.963      0.608
Speed: 0.3ms preprocess, 7.0ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /home/kien/Desktop/projects/Amir_research/yolo/runs/detect/val
Ultralytics 8.3.195 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3080, 9864MiB)
Model summary (fused): 72 layers, 3,006,428 parame

In [5]:
df = df.drop(columns=["model"])
df["map50_95"] = [result.box.map for result in results]
df["speed"] = [sum(result.speed.values()) for result in results]
df["s_per_pic"] = df.speed/270
df["fps"] = 1/df["s_per_pic"]

In [6]:
df

,gen,name,map50_95,speed,s_per_pic,fps
0,yolo8,yolov8m,0.607781,8.612888,0.031900,31.348368
1,yolo8,yolov8n,0.599128,6.662799,0.024677,40.523512
2,yolo8,yolov8l,0.616803,10.406087,0.038541,25.946353
3,yolo8,yolov8x,0.626444,16.938544,0.062735,15.939977
4,yolo8,yolov8s,0.598078,6.206764,0.022988,43.500931
5,yolo11,yolo11l,0.618033,19.420865,0.071929,13.902573
6,yolo11,yolo11n,0.590898,13.486247,0.049949,20.020395
7,yolo11,yolo11s,0.602350,9.198367,0.034068,29.353037
8,yolo11,yolo11m,0.612337,9.507070,0.035211,28.399918
9,yolo11,yolo11x,0.592882,20.327080,0.075285,13.282774


In [7]:
df.to_csv("./runs/val_results.csv", index=False)